# 详细文档: 水文模型 - 汇流模块对比

**相关模块:** `hydro_model/routing.py`

## 目的
此文档旨在详细介绍`hydro_model`中包含的多种汇流模块，并对它们进行全面的对比。汇流模块是水文模型的另一个核心，它负责将产流模块计算出的地表径流（净雨），通过河道的槽蓄和演算，转化为流域出口的流量过程线。

此笔记本将：
1.  简要介绍`SimpleRouting`, `MuskingumRouting`, `UnitHydrographRouting`, `MuskingumCungeRouting` 四种模块的基本原理。
2.  使用一个相同的入流过程线，分别通过这四个模块进行演算。
3.  将四个模块的演算结果绘制在同一张图上，直观地比较它们对洪水波的削减和延迟效果。

## 1. 环境设置

首先，我们导入所需的库和我们自己的汇流模块。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from hydro_model.routing import SimpleRouting, MuskingumRouting, UnitHydrographRouting, MuskingumCungeRouting

## 2. 准备输入数据

我们创建一个人工的、三角形的入流过程线，它将作为所有汇流模块的共同输入。

In [ ]:
inflow_hydrograph = np.array([0, 10, 20, 40, 60, 45, 30, 20, 10, 5, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0], dtype=float)
timesteps = np.arange(len(inflow_hydrograph))

## 3. 运行各个汇流模块

我们依次实例化并运行四种不同的汇流模块。

In [ ]:
# --- 模块1: SimpleRouting ---
# 原理: 将入流分为快、慢两个线性水库进行演算。
simple_router = SimpleRouting(k_q=0.7, k_s=0.2)
outflow_simple = [simple_router.run(q) for q in inflow_hydrograph]

# --- 模块2: MuskingumRouting ---
# 原理: 经典马斯京根法，基于经验参数K和x。
muskingum_router = MuskingumRouting(K=2.0, x=0.25)
outflow_muskingum = [muskingum_router.run(q) for q in inflow_hydrograph]

# --- 模块3: UnitHydrographRouting ---
# 原理: 将入流（视为净雨）与单位线进行卷积。
uh_ordinates = np.array([0.1, 0.4, 0.3, 0.15, 0.05])
uh_router = UnitHydrographRouting(uh_ordinates=uh_ordinates)
outflow_uh = [uh_router.run(q) for q in inflow_hydrograph]

# --- 模块4: MuskingumCungeRouting ---
# 原理: 基于物理参数的马斯京根-康基法，动态计算K和x。
mc_router = MuskingumCungeRouting(length=5000, slope=0.001, manning_n=0.03, width=50, dt=1.0)
outflow_mc = [mc_router.run(q) for q in inflow_hydrograph]

print("所有汇流模块运行完毕。")

## 4. 结果对比与可视化

现在我们将原始入流和四个模块的演算结果绘制在同一张图上，进行对比。

In [ ]:
plt.figure(figsize=(15, 8))

plt.plot(timesteps, inflow_hydrograph, 'k--', linewidth=2, label='Inflow Hydrograph')
plt.plot(timesteps, outflow_simple, 'b-o', markersize=4, label='SimpleRouting')
plt.plot(timesteps, outflow_muskingum, 'r-s', markersize=4, label='Muskingum (K=2, x=0.25)')
plt.plot(timesteps, outflow_uh, 'g-^', markersize=4, label='Unit Hydrograph')
plt.plot(timesteps, outflow_mc, 'm-d', markersize=4, label='Muskingum-Cunge')

plt.title('Comparison of Different Routing Modules')
plt.xlabel('Time Step (days)')
plt.ylabel('Flow (unit depends on input)')
plt.legend()
plt.grid(True, linestyle='--')
plt.show()

### 对比分析

从上图可以清晰地看到不同汇流算法的特点：
- **削峰作用**: 所有汇流方法都对洪峰（inflow hydrograph的峰值）起到了削减作用，这是河道槽蓄的典型效应。其中，`Muskingum`和`Muskingum-Cunge`的削峰效果最符合物理直觉。
- **延迟作用**: 所有方法都使洪峰出现的时间向后延迟。`Muskingum`和`Muskingum-Cunge`的延迟效应也最为明显。
- **单位线法**: `UnitHydrograph`的结果显示出多个波峰，这是因为它的输入被视为连续的净雨脉冲，每个脉冲都会生成一个按单位线缩放和时移的径流过程，最终结果是这些过程的叠加。这与其他将输入视为“流量”的方法在概念上有所不同。
- **简单汇流法**: `SimpleRouting`也展示了一定的削峰和延迟，但其物理意义不如其他方法明确，更像是一个概念性的分割和延迟模型。

在实际应用中，应根据模型的目的、数据的可用性和对物理过程的保真度要求来选择合适的汇流模块。